In [1]:
import os
import pandas as pd

In [2]:
import os

# 확인할 디렉토리 경로
path = '../data/raw/apt_images'


# 파일만 카운트
file_count = len([name for name in os.listdir(path)
                  if os.path.isfile(os.path.join(path, name))])

print(f'파일 개수: {file_count}')

파일 개수: 23396


In [3]:
df = pd.read_csv("../data/interim/apt/apt_with_long_lat.csv")

In [4]:
len(df)

23573

In [5]:
df.dropna(inplace=True)

In [6]:
len(df)

23395

In [16]:
df = df[['위도','경도']]

In [18]:
df.rename(columns={'위도': 'latitude', '경도': 'longitude'}, inplace=True)

In [22]:
df.head()

,latitude,longitude
0,37.659925,127.076754
1,37.561195,126.956170
2,37.467098,127.102102
3,37.642869,127.059507
4,37.643712,127.053952


In [27]:
import pandas as pd
import pymysql

try:
    conn = pymysql.connect(
    host='localhost',
    user='root',
    password='As589788@@',
    db='apt_price',
    charset='utf8mb4',
    cursorclass=pymysql.cursors.DictCursor
)

    cursor = conn.cursor()

    sql = "INSERT INTO lon_lat (latitude, longitude) VALUES (%s, %s)"
    data = list(df[['latitude', 'longitude']].itertuples(index=False, name=None))

    cursor.executemany(sql, data)
    conn.commit()
    cursor.close()
    conn.close()
    print("CSV 데이터 삽입 완료")
except Exception as e:
    print(f"CSV 삽입 오류: {e}")

CSV 데이터 삽입 완료


In [26]:
import os
import re

# 파일들이 있는 폴더 경로
folder_path = '../data/raw/apt_images'

# 파일명에서 숫자만 추출하는 정규표현식
pattern = re.compile(r'apt_image_(\d+)\.jpg')

numbers = []

for filename in os.listdir(folder_path):
    match = pattern.match(filename)
    if match:
        num = int(match.group(1))
        numbers.append(num)

if not numbers:
    print("해당 패턴의 파일이 없습니다.")
    exit()

numbers.sort()

# 중복된 번호 찾기
duplicates = set([x for x in numbers if numbers.count(x) > 1])

# 누락된 번호 찾기
full_range = set(range(numbers[0], numbers[-1] + 1))
missing = full_range - set(numbers)

print(f"총 파일 개수: {len(numbers)}")
print(f"중복된 번호: {sorted(duplicates)}")
print(f"누락된 번호: {sorted(missing)}")

총 파일 개수: 23395
중복된 번호: []
누락된 번호: []


# 뉴스 기사

In [1]:
import pandas as pd
from gensim import corpora, models
import networkx as nx
from collections import defaultdict
import jpype, os, json
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
import numpy as np
import numpy as np
from tqdm import tqdm

In [2]:
KINDS_PATH = '../data/interim/news/kinds_news.csv'
kinds = pd.read_csv(KINDS_PATH)
df = kinds

docs= list(df['content'])
print(f"수집된 기사의 길이 : {len(docs)}")

수집된 기사의 길이 : 78751


In [3]:
STOP_WORDS_PATH = '../data/raw/news/stopwords-ko.txt'

with open(STOP_WORDS_PATH, 'r', encoding='utf-8') as f:
    stopwords = [line.strip() for line in f if line.strip()]  # 공백 제거 + 빈 줄 제거

print(f"불용어의 갯수 : {len(stopwords)}")

불용어의 갯수 : 1453


In [4]:
SENTIMENT_DIC_PATH = '../data/raw/news/SentiWord_info.json'
with open(SENTIMENT_DIC_PATH, 'r', encoding='utf-8') as f:
    senti_dict_list = json.load(f)

senti_dic = {item['word']: int(item['polarity']) for item in senti_dict_list}

print(f"감성 사전의 길이 : {len(senti_dic)}")

감성 사전의 길이 : 14852


In [6]:
# 1. 전처리 ( 형태소 분석 )
from konlpy.tag import Mecab
mecab = Mecab()
texts = [[w for w in mecab.morphs(doc) if len(w) > 1] for doc in docs]

In [9]:
len(texts)

78751

In [11]:
# 불용어 제거

cleaned_token_lists = [
    [w for w in tokens if w not in stopwords and w.isalpha()]
    for tokens in texts
]

print(cleaned_token_lists[0])

['신고', '신고', '신고', '과거', '패턴', '집값', '격차', '과거', '양극', '집값', '상승', '주목', '부동산', '거래', '분석']


# LDA 토픽 모델링

In [13]:
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in cleaned_token_lists]
lda_model = models.LdaModel(corpus, num_topics=8, id2word=dictionary, passes=10)

topics = lda_model.print_topics(num_words=30)

In [15]:
print(f"주요 토픽 예시 : {topics[0]}")

주요 토픽 예시 : (0, '0.024*"전용" + 0.019*"아파트" + 0.018*"면적" + 0.016*"오피스텔" + 0.015*"주거" + 0.013*"분양" + 0.012*"부동산" + 0.012*"시설" + 0.012*"코로나" + 0.011*"생활" + 0.011*"상가" + 0.010*"시장" + 0.009*"지역" + 0.009*"지상" + 0.009*"가구" + 0.008*"최근" + 0.008*"지하" + 0.008*"감염증" + 0.008*"브랜드" + 0.007*"수익" + 0.007*"규모" + 0.007*"수요" + 0.007*"상업" + 0.006*"인기" + 0.006*"지식" + 0.006*"면서" + 0.006*"세대" + 0.006*"오피스" + 0.006*"상품" + 0.006*"센터"')


# LDA에서 추출된 주요 단어 집합 생성

In [16]:
lda_keywords = set()
for topic in lda_model.show_topics(num_topics=8, num_words=30, formatted=False):
    for word, _ in topic[1]:
        lda_keywords.add(word)

In [17]:
print("LDA 키워드 후보 : ", lda_keywords)

LDA 키워드 후보 :  {'은행', '합니다', '지금', '금리', '감염증', '복합', '보증금', '빌딩', '매물', '홍남기', '투자', '업계', '세입자', '보다', '주거', '안정', '기록', '관리', '관련', '매입', '국내', '방안', '지상', '개발', '면적', '가격', '법인', '부채', '김현미', '조성', '조합', '회사', '시중', '올해', '자금', '다고', '가계', '상품', '전세', '공시', '재정부', '정책', '공인', '라고', '재산세', '공급', '문제', '부담', '오피스텔', '상가', '구역', '오늘', '밝혔', '정비', '전셋값', '임대', '때문', '지하', '기업', '종합부동산세', '장관', '상황', '지난달', '생활', '브랜드', '한다', '대출', '분양', '증여', '주식', '경매', '국회', '나오', '정보', '규제', '신용', '취득세', '시설', '주택', '인기', '생각', '지식', '규모', '계약', '민간', '코로나', '사업', '일대', '미국', '지역', '투기', '임대차', '정도', '면서', '국세청', '재산', '기획', '직원', '수익', '지난해', '담보', '토지', '분석', '가장', '호재', '예정', '많이', '추진', '문재인', 'LH', '거래', '전용', '통계', '위원회', '공공', '아파트', '센터', '상업', '수요', '갱신', '청약', '집값', '전월세', '평균', '건설', '보유', '시행', '수도', '물량', '금융', '사람', '이날', '적용', '자산', '대한', '당국', '국민', '상승', '재건축', '는데요', '공사', '부지', '건축', '가구', '민주당', '시장', '매매', '라는', '월세', '선정', '의원', '결과', '세대', '지구', '직장', '최근', '도시', '세금', '리츠', '오

# 해당 단어가 포함된 문장 추출

In [18]:
keyword_sentences = defaultdict(list)
for doc in docs:
    for word in lda_keywords:
        if word in doc:
            keyword_sentences[word].append(doc)